# Checkpoint de teoria: RPS e Baseline

**Fase 0: dados e baseline** · projeto `brasileirao_forecast`

Este notebook explica as duas peças que dão régua a todo o projeto: a métrica **RPS** e o **Baseline**. Se este é seu primeiro contato com o RPS, comece pela **seção 2.6**, um reforço do zero que roda sem os dados. Rode célula por célula e, onde houver o comentário `# <- altere`, mude os valores e veja o resultado reagir.

**Antes de rodar:** baixe o `BRA.csv` em [football-data.co.uk/brazil.php](https://football-data.co.uk/brazil.php) e salve em `data/raw/BRA.csv` (a pasta `data/` não é versionada). Termos de uso e ressalvas dessa fonte: [`docs/sources.md`](../../docs/sources.md).

## 1. Pergunta

O **Alvo** do projeto é dar as probabilidades de vitória, empate e derrota do Vasco na próxima partida do Brasileirão. Antes de construir qualquer modelo, duas perguntas precisam de resposta:

1. Como medir se um conjunto de previsões probabilísticas é bom?
2. Qual é o mínimo que um modelo precisa superar para valer a pena?

## 2. Ideia central

### 2.1 Previsão probabilística

Em vez de dizer "o Vasco vence", o modelo diz algo como *55% vitória, 25% empate, 20% derrota*. Isso é mais honesto sobre a incerteza do futebol, mas exige uma régua que premie probabilidades bem calibradas, e não só acertos. Essas réguas se chamam *scoring rules* (regras de pontuação).

Convenção do projeto: as probabilidades vêm sempre na ordem **mandante vence, empate, visitante vence** abreviadas como `H` (*home win*), `D` (*draw*) e `A` (*away win*).

### 2.2 RPS (Ranked Probability Score)

Cada jogo vira duas perguntas de sim ou não, em ordem:

1. O mandante vence?
2. O mandante vence **ou** empata?

O modelo responde com probabilidades acumuladas: `p_home` e `p_home + p_draw`. O RPS é a média dos erros ao quadrado nessas duas perguntas:

$$\text{RPS} = \frac{1}{r-1}\sum_{k=1}^{r-1}\left(\sum_{j=1}^{k}(p_j - o_j)\right)^2$$

com `r = 3` resultados, `p_j` a probabilidade prevista para o resultado `j` e `o_j` igual a 1 se esse resultado aconteceu e 0 caso contrário. O valor vai de 0 (perfeito) a 1 (certeza total e errada). **Menor é melhor.** A nota do modelo é a média do RPS sobre os jogos. Se a fórmula parecer pesada, a seção 2.6 refaz tudo do zero, com legenda, receita em passos e exercícios.

Detalhe importante: a ordem importa. Errar por *dois degraus* (previu vitória do mandante, o visitante ganhou) custa mais do que errar por *um* (previu vitória, deu empate).

### 2.3 Por que não só acurácia?

Acurácia só vê se o resultado mais provável aconteceu e ignora a confiança do modelo. Log-loss e Brier score usam as probabilidades, mas tratam vitória, empate e derrota como categorias sem ordem. Por isso o RPS é a métrica primária do projeto, com os outros dois como apoio.

### 2.4 Baseline

É o palpite de quem não sabe nada sobre os times: as frequências históricas de cada resultado, iguais para todo jogo. Ele já embute uma informação real (a vantagem de jogar em casa), mas nenhuma sobre *quem* joga. Serve de régua (um modelo só vale se superá-lo) e de detector de bugs (se um modelo sofisticado perde dele, há algo errado).

### 2.5 Validação walk-forward

Para cada dia de jogos, o modelo é ajustado **só com jogos de dias anteriores** e prevê os jogos do dia. É assim que a previsão funcionará na vida real, e evita o vazamento de informação: usar o futuro para prever o passado.

### 2.6 Reforço: entendendo o RPS do zero

Esta seção é para o primeiro contato com o RPS. Ela **não precisa dos dados**: dá para rodar tudo aqui sem o `BRA.csv`.

#### Legenda: o que são H, D e A

| Símbolo | Em inglês | Em português |
| --- | --- | --- |
| `H` | *Home win* | vitória do **mandante** (time da casa) |
| `D` | *Draw* | **empate** |
| `A` | *Away win* | vitória do **visitante** (time de fora) |

As probabilidades vêm sempre nessa ordem, `(p_H, p_D, p_A)`, que no código são as colunas `p_home`, `p_draw` e `p_away`. Por exemplo, a previsão `(0,2 / 0,3 / 0,5)` quer dizer: 20% de vitória do mandante, 30% de empate e 50% de vitória do visitante.

Cada jogo termina em um só resultado, que escrevemos como um vetor de zeros e um: `o = (1, 0, 0)` se foi `H`, `o = (0, 1, 0)` se foi `D` e `o = (0, 0, 1)` se foi `A`.

#### Do Brier ao RPS

Uma primeira ideia seria somar o erro ao quadrado de cada resultado (é o Brier score). Mas ele não sabe que o empate fica *no meio* entre as duas vitórias. Veja dois modelos que dão 50% à vitória do mandante, que foi o que aconteceu. O X guarda o resto da dúvida no empate; o Y joga o resto na vitória do visitante.

In [ ]:
import numpy as np
import pandas as pd

from brasileirao_forecast.metrics import brier_score, log_loss, rps

models = {
    "X (0,5 / 0,4 / 0,1)": [0.5, 0.4, 0.1],  # dúvida no empate, perto de H
    "Y (0,5 / 0,1 / 0,4)": [0.5, 0.1, 0.4],  # dúvida na vitória do visitante, longe de H
}

pd.DataFrame(
    {
        name: {
            "Brier": brier_score([p], ["H"]),
            "log-loss": log_loss([p], ["H"]),
            "RPS": rps([p], ["H"]),
        }
        for name, p in models.items()
    }
).T

Brier e log-loss dão o mesmo valor aos dois modelos. Só o RPS percebe que o Y colocou sua dúvida no resultado mais distante do que aconteceu.

#### A fórmula, reforçada

O RPS transforma o jogo em **perguntas acumuladas, na ordem H, D, A**:

- **Pergunta 1:** o mandante vence? (`H`) Previsto: `p_H`. Aconteceu: `o_H`.
- **Pergunta 2:** o mandante vence **ou** empata? (`H` ou `D`) Previsto: `p_H + p_D`. Aconteceu: `o_H + o_D`.
- (Uma pergunta 3, "aconteceu H, D ou A?", vale sempre 100% previsto e 100% observado, então o erro é zero e ela não entra. Com `r` resultados há `r − 1` perguntas.)

**Para um jogo com três resultados**, a fórmula é:

$$\text{RPS} = \frac{1}{2}\Big[(p_H - o_H)^2 + \big((p_H + p_D) - (o_H + o_D)\big)^2\Big]$$

Em palavras: *erro da pergunta 1 ao quadrado, mais erro da pergunta 2 ao quadrado, tudo dividido por 2.*

**A versão geral**, para `r` resultados ordenados, é a que vimos em 2.2:

$$\text{RPS} = \frac{1}{r-1}\sum_{k=1}^{r-1}\Big(\sum_{j=1}^{k}(p_j - o_j)\Big)^2$$

Lendo de dentro para fora:

- `Σ_{j≤k} (p_j − o_j)` é a diferença entre o acumulado previsto e o acumulado observado até o degrau `k`.
- Elevar ao quadrado faz erros grandes pesarem muito mais, e o sinal do erro não importa.
- A soma externa percorre os degraus `k = 1, …, r − 1`.
- Dividir por `r − 1` dá a média por pergunta e mantém o RPS entre 0 e 1.

A nota de um modelo é a **média do RPS sobre todos os jogos** avaliados.

#### Receita em quatro passos

1. Escreva a previsão na ordem `(p_H, p_D, p_A)` e o resultado como vetor `o`.
2. Calcule os acumulados da previsão e do resultado, e descarte o último de cada (sempre 1).
3. Subtraia, nos dois degraus: acumulado previsto menos acumulado observado.
4. Eleve cada diferença ao quadrado, some as duas e divida por 2.

#### Exemplo resolvido: previsão (0,2 / 0,3 / 0,5), resultado H

| | Pergunta 1 (H?) | Pergunta 2 (H ou D?) |
| --- | --- | --- |
| Previsão acumulada | 0,2 | 0,2 + 0,3 = 0,5 |
| Observado acumulado (H = 1, 0, 0) | 1 | 1 |
| Erro (previsto − observado) | 0,2 − 1 = −0,8 | 0,5 − 1 = −0,5 |
| Erro ao quadrado | 0,64 | 0,25 |

RPS = (0,64 + 0,25) / 2 = **0,445**. O mandante venceu, mas o modelo dava só 20% a essa vitória e só 50% a "H ou D": errou as duas perguntas por muito.

A função abaixo faz esse passo a passo para qualquer previsão. Vamos reutilizá-la mais adiante.

In [ ]:
def rps_step_by_step(probs, outcome):
    """Print the two cumulative questions behind the RPS for one match."""
    observed = {"H": (1, 0, 0), "D": (0, 1, 0), "A": (0, 0, 1)}[outcome]
    cum_forecast = np.cumsum(probs)[:2]
    cum_observed = np.cumsum(observed)[:2]
    gaps = cum_forecast - cum_observed
    value = (gaps**2).sum() / 2
    print(f"Aconteceu: {outcome}")
    print(f"  Pergunta 1 (mandante vence?):           previsto {cum_forecast[0]:.2f} | "
          f"aconteceu {cum_observed[0]} | erro {gaps[0]:+.2f}")
    print(f"  Pergunta 2 (mandante vence ou empata?): previsto {cum_forecast[1]:.2f} | "
          f"aconteceu {cum_observed[1]} | erro {gaps[1]:+.2f}")
    print(f"  RPS = ({gaps[0]**2:.3f} + {gaps[1]**2:.3f}) / 2 = {value:.3f}\n")
    return value


value = rps_step_by_step([0.2, 0.3, 0.5], "H")

#### Por que a nota é honesta

O RPS é uma regra de pontuação **própria**: a melhor estratégia é sempre dizer o que você realmente acredita. Veja a conta mais simples, no caso de sim ou não. Se a chance verdadeira de vitória fosse 60%:

- Reportar 0,6 dá erro esperado de 0,6 × 0,16 + 0,4 × 0,36 = **0,24**.
- Reportar 0,9 (exagerar a confiança) dá 0,6 × 0,01 + 0,4 × 0,81 = **0,33**, pior.

O RPS é feito dessas mesmas contas nas perguntas acumuladas, e herda a propriedade. Por isso ele pune o excesso de confiança e premia a honestidade.

#### Como interpretar o número

- **0** é perfeito e **1** é certeza total e errada. No futebol ninguém chega perto de 0, porque o jogo é muito incerto.
- Neste projeto, o palpite uniforme (1/3 para cada) dá cerca de 0,233 e o Baseline dá cerca de 0,218. Os modelos vão disputar a faixa abaixo disso.
- Um RPS isolado não diz nada. Só vale comparado com outro modelo **nos mesmos jogos**, e o Baseline é a comparação mínima.

#### Equívocos comuns

- **"RPS é acurácia."** Não: o RPS usa a confiança do modelo, e dois modelos com a mesma acurácia podem ter RPS bem diferentes (veja 5.3).
- **"RPS menor prova que o modelo é melhor."** Só se a diferença for maior que o ruído da amostra (veja 5.9).
- **"RPS baixo significa probabilidades calibradas."** Não necessariamente. Calibração é uma pergunta à parte, que vamos olhar com uma curva mais adiante.

#### Exercícios (calcule na mão antes de rodar!)

Lembre a legenda: `H` = mandante vence, `D` = empate, `A` = visitante vence. Em cada exercício, escreva sua conta em um papel e depois digite o resultado na célula de código no lugar de `None`. A célula diz se você acertou.

**Exercício 1.** Previsão `(0,2 / 0,3 / 0,5)`, isto é, 20% `H`, 30% `D` e 50% `A`. Calcule o RPS se o resultado for `A`, depois se for `H`, depois se for `D`.

In [ ]:
def check(my_answer, correct):
    if my_answer is None:
        return "sem resposta ainda"
    if np.isclose(my_answer, correct, atol=0.001):
        return "certo"
    return "diferente: use rps_step_by_step para comparar com a sua conta"


exercise_1_forecast = [0.2, 0.3, 0.5]  # [H, D, A]

# <- digite aqui o RPS que você calculou para cada resultado
my_answers_1 = {"A": None, "H": None, "D": None}

for outcome, mine in my_answers_1.items():
    correct = rps([exercise_1_forecast], [outcome])
    print(f"Resultado {outcome}: {check(mine, correct)}")

**Exercício 2.** O jogo terminou em empate (`D`). Qual previsão é melhor: **P1 = (0,3 / 0,4 / 0,3)** ou **P2 = (0,45 / 0,10 / 0,45)**? Calcule o RPS das duas e diga qual ganha.

In [ ]:
forecasts_2 = {"P1": [0.3, 0.4, 0.3], "P2": [0.45, 0.10, 0.45]}

# <- digite aqui o RPS que você calculou para cada previsão
my_answers_2 = {"P1": None, "P2": None}

for name, mine in my_answers_2.items():
    correct = rps([forecasts_2[name]], ["D"])
    print(f"{name}: {check(mine, correct)}")

**Exercício 3 (sem calcular).** Para a previsão `(0,5 / 0,3 / 0,2)`, o RPS é igual quando o resultado é `H` e quando é `D`, mas bem maior quando é `A`. Por quê? A célula abaixo mostra os três valores, e a explicação você escreve em seguida.

In [ ]:
exercise_3_forecast = [0.5, 0.3, 0.2]

pd.Series(
    {outcome: rps([exercise_3_forecast], [outcome]) for outcome in ["H", "D", "A"]},
    name="RPS se o resultado for...",
)

*Minha resposta ao exercício 3:*

*(escreva aqui, com suas palavras, antes de olhar as respostas abaixo.)*

---

#### Respostas (leia depois de tentar)

1. Resultado `A`: **0,145**. Resultado `H`: **0,445**. Resultado `D`: **0,145**. Repare que é o espelho da previsão `(0,5 / 0,3 / 0,2)`, com mandante e visitante trocados.
2. P1: **0,090**. P2: **0,2025**. P1 ganha por bastante: P2 deu só 10% ao empate, que foi o que aconteceu, e dividiu o resto entre as duas vitórias.
3. Com `H` ou com `D`, apenas uma das duas perguntas acumuladas é muito errada. Com `A`, o modelo erra **as duas ao mesmo tempo**: o visitante vencer contraria "o mandante vence?" e também "o mandante vence ou empata?". Errar por dois degraus pesa mais que errar por um.

### 2.7 Para ler depois

Confira os dados exatos das publicações antes de citá-las:

- Epstein (1969): proposta original do RPS, em meteorologia.
- Constantinou e Fenton (2012): regras de pontuação para modelos de previsão de futebol.

## 3. Suposições

- **A ordem vitória, empate, derrota é uma convenção.** O RPS assume que o empate fica "entre" as duas vitórias. É razoável no futebol (pense no saldo de gols), mas é uma escolha.
- **O Baseline assume jogos intercambiáveis.** A mesma vantagem de mando para todos os clubes e constante ao longo do tempo.
- **A avaliação assume que o passado representa o futuro.** Jogos de 2013 e de 2026 pesam igual no histórico, embora o futebol mude.
- **Os resultados chegam sem atraso.** Cada dia de jogos é previsto com tudo o que aconteceu nos dias anteriores.

## 4. Onde falha

- **Amostra pequena engana.** O RPS do Vasco usa cerca de 370 jogos, e a diferença para o RPS geral pode ser só ruído (veja 5.9).
- **O Baseline ignora os times.** Dá o mesmo palpite para Flamengo x Vasco e para um recém-promovido jogando fora. Esse é o espaço de melhora dos modelos.
- **O RPS não compara ligas.** Só vale comparar modelos nos *mesmos* jogos.
- **Os dados têm falhas.** O arquivo traz um jogo de 2016 sem placar (descartado pelo carregador) e alguns horários com espaço sobrando.
- **Espiar o futuro quase não muda o Baseline** (veja 5.8), porque as frequências são estáveis. Com modelos flexíveis o estrago é enorme, por isso a disciplina do walk-forward vale desde já.

## 5. Resultados e experimentos

### 5.0 Carregando os dados

In [ ]:
import numpy as np
import pandas as pd

from brasileirao_forecast.baseline import FrequencyBaseline
from brasileirao_forecast.evaluation import matches_of, walk_forward_predictions
from brasileirao_forecast.loaders import load_results
from brasileirao_forecast.metrics import PROB_COLUMNS, rps, score_by_season, score_predictions
from brasileirao_forecast.paths import RAW_DIR

CSV_PATH = RAW_DIR / "BRA.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Baixe o BRA.csv em football-data.co.uk/brazil.php e salve em {CSV_PATH}"
    )

results = load_results(CSV_PATH)
FOCUS_TEAM = "Vasco"
print(
    f"{len(results)} partidas, de {results['kickoff'].min():%d/%m/%Y} "
    f"a {results['kickoff'].max():%d/%m/%Y}"
)

### 5.1 O RPS na mão

A função `rps_step_by_step`, definida na seção 2.6, refaz o cálculo passo a passo para um jogo. Aqui ela confere com a função do projeto (`assert`). Altere `forecast` (casa, empate, fora, somando 1) e rode de novo.

In [ ]:
forecast = [0.5, 0.3, 0.2]  # <- altere: [casa, empate, fora]

for outcome in ["H", "D", "A"]:
    by_hand = rps_step_by_step(forecast, outcome)
    assert np.isclose(by_hand, rps([forecast], [outcome]))

### 5.2 A ordem importa

Imagine um modelo 100% certo de que o mandante vence. Quanto ele paga em cada resultado?

In [ ]:
certain_home = [1.0, 0.0, 0.0]  # <- altere
pd.Series(
    {outcome: rps([certain_home], [outcome]) for outcome in ["H", "D", "A"]},
    name="RPS se o resultado for...",
)

Com `certain_home = [1.0, 0.0, 0.0]`, o empate custa 0,5 e a vitória do visitante custa 1,0: errar por dois degraus pesa o dobro de errar por um. Tente trocar `certain_home` por `[0.0, 1.0, 0.0]` (certeza no empate) e veja que errar para qualquer lado custa o mesmo.

### 5.3 Acurácia contra RPS

Dois modelos que sempre apostam na vitória do mandante, em quatro jogos (H, H, D, A). Um é muito confiante, o outro é moderado. Ambos acertam o mesmo número de jogos.

In [ ]:
outcomes = ["H", "H", "D", "A"]
models = {
    "confiante (0,8 / 0,1 / 0,1)": [[0.8, 0.1, 0.1]] * 4,
    "moderado (0,5 / 0,3 / 0,2)": [[0.5, 0.3, 0.2]] * 4,
}
labels = ["H", "D", "A"]

rows = {}
for name, probs in models.items():
    hits = np.mean([labels[int(np.argmax(p))] == o for p, o in zip(probs, outcomes)])
    rows[name] = {"acurácia": hits, "RPS": rps(probs, outcomes)}
pd.DataFrame(rows).T

A acurácia é igual (50%), mas o modelo moderado tem RPS menor: o confiante foi punido com força pelo empate e pela vitória do visitante. É por isso que a acurácia sozinha engana.

### 5.4 O Baseline: as frequências históricas

In [ ]:
counts = results["result"].value_counts().reindex(["H", "D", "A"])
pd.DataFrame({"jogos": counts, "frequência": counts / counts.sum()})

### 5.5 O Baseline em walk-forward

Cada jogo é previsto só com as frequências dos dias anteriores. A temporada de 2012 serve apenas de histórico; a avaliação começa em 2013.

In [ ]:
FIRST_TEST_SEASON = 2013

predictions = walk_forward_predictions(results, FrequencyBaseline(), FIRST_TEST_SEASON)

pd.DataFrame(
    {
        "todas as partidas": score_predictions(predictions),
        f"jogos do {FOCUS_TEAM}": score_predictions(matches_of(predictions, FOCUS_TEAM)),
    }
).T

### 5.6 Por temporada

In [ ]:
score_by_season(predictions)

### 5.7 Quanto o Baseline melhora sobre o palpite uniforme?

O **skill score** mede quanto do erro de uma referência o modelo elimina: `1 - RPS do modelo / RPS da referência`.

In [ ]:
uniform_rps = rps(np.full((len(predictions), 3), 1 / 3), predictions["result"].to_numpy())
baseline_rps = score_predictions(predictions)["rps"]

print(f"RPS do palpite uniforme (1/3 cada): {uniform_rps:.4f}")
print(f"RPS do Baseline:                    {baseline_rps:.4f}")
print(f"Skill do Baseline sobre o uniforme: {1 - baseline_rps / uniform_rps:.1%}")

### 5.8 Espiar o futuro

E se o Baseline fosse calculado com **todos** os jogos, inclusive os que ainda não aconteceram quando cada partida seria prevista? Isso é vazamento de informação.

In [ ]:
peeking = FrequencyBaseline().fit(results)  # usa TODOS os jogos, inclusive futuros
test = results[results["season"] >= FIRST_TEST_SEASON]
peeking_rps = rps(peeking.predict(test), test["result"].to_numpy())

print(f"Baseline honesto (walk-forward): {baseline_rps:.5f}")
print(f"Baseline espiando o futuro:      {peeking_rps:.5f}")
print(f"Diferença:                       {peeking_rps - baseline_rps:+.5f}")

A diferença é minúscula, porque as frequências quase não mudam de uma temporada para outra. Mas o Baseline só tem **três números** para memorizar. Um modelo com um parâmetro por clube, ajustado ao histórico inteiro, memorizaria o futuro e pareceria ótimo sem ser. O walk-forward existe para impedir isso.

### 5.9 Quanto ruído há no RPS do Vasco?

O RPS dos jogos do Vasco ficou um pouco acima do geral. Será que isso significa algo? Para estimar a incerteza, o *bootstrap* sorteia, com reposição, jogos do Vasco milhares de vezes e recalcula a média a cada sorteio.

In [ ]:
def per_match_rps(df):
    """RPS of each match (same formula as the library, one value per row)."""
    probs = df[PROB_COLUMNS].to_numpy()
    observed = np.eye(3)[["HDA".index(r) for r in df["result"]]]
    return (np.cumsum(probs - observed, axis=1)[:, :2] ** 2).sum(axis=1) / 2


vasco = matches_of(predictions, FOCUS_TEAM)
vasco_rps = per_match_rps(vasco)
assert np.isclose(vasco_rps.mean(), score_predictions(vasco)["rps"])

rng = np.random.default_rng(42)
bootstrap_means = [rng.choice(vasco_rps, size=len(vasco_rps)).mean() for _ in range(2000)]
low, high = np.percentile(bootstrap_means, [2.5, 97.5])

print(f"RPS do {FOCUS_TEAM}: {vasco_rps.mean():.4f}  (intervalo de 95%: {low:.4f} a {high:.4f})")
print(f"RPS geral:          {baseline_rps:.4f}")
print("O geral cai dentro do intervalo?", low <= baseline_rps <= high)

Se o RPS geral cai dentro do intervalo do Vasco, **não dá para afirmar** que o time é mais difícil de prever que os outros: a diferença cabe no ruído de uma amostra de cerca de 370 jogos. O mesmo cuidado vale ao comparar modelos: uma melhora pequena no RPS pode ser só sorte da amostra.

## 6. O que aprendi e o que ficou em aberto

Com dados até 20/09/2026:

- O Baseline tem RPS de cerca de **0,218** (2013 a 2026), contra cerca de 0,233 do palpite uniforme. Saber a frequência geral dos resultados reduz o erro em apenas cerca de 6,5%. **Esse 0,218 é a régua** que os modelos das próximas fases precisam superar.
- O RPS do Vasco (cerca de 0,222) fica dentro do intervalo de incerteza em que o geral também cai, então não há evidência de que o Vasco seja mais difícil de prever.
- Calcular o Baseline espiando o futuro mudou o RPS em cerca de 0,0001. Para ele é irrelevante, mas a disciplina do walk-forward é obrigatória para modelos que aprendem por clube.

**Em aberto (para as próximas fases):**

- Um Elo simples já é um Baseline bem mais exigente que as frequências? (Fase 1)
- Quão grande precisa ser a melhora no RPS para ser real e não sorte da amostra? Isso pede comparar modelos jogo a jogo, com intervalo de confiança da diferença.
- As probabilidades estão **calibradas**, isto é, quando o modelo diz 60%, acontece cerca de 60% das vezes? O RPS não responde sozinho; vale uma curva de calibração.
- Como tratar clubes promovidos, que chegam sem histórico na Série A?

### Minhas anotações

*(Escreva aqui, com suas palavras, o que ficou claro e o que ainda não.)*